In [32]:
import os
import pandas
from utils import partial_match_scores

dataset_root = "/home/xzhao/workspace/GYB_self-ensemble/datasets"
ds_name = "myriadlama"
# model_name = "llama3.2_3b"
model_name = "qwen2.5_7b_it"

def get_filenames(
        modifyattn, modifyrope, scale_score,
        repeat_paras, single_para_qapair, 
        num_paraphrases, num_samples=5):
    dump_file = f"{dataset_root}/{ds_name}/{model_name}/myriadlama."
    if modifyattn:
        dump_file += "modifyattn."
    if modifyrope:
        dump_file += "modifyrope."
    if repeat_paras:
        dump_file += "repeatparas."
    if scale_score:
        dump_file += "scalescore20."
    
    if single_para_qapair:
        dump_file += "singleparaqapair."


    dump_file += f"{num_samples}samples.{num_paraphrases}paras.feather"
    # print(f"Loading from {dump_file}")
    return dump_file

def _calculate_accuracy(df, label):
    predicts = [[pred] for pred in df["predict_lemma"].tolist()]
    answers = [answers for answers in df["answer_lemmas"]]
    acc = partial_match_scores(predicts, answers, birdirect=True)    
    print(f"{label} Accuracy: {acc:.4f}")

def calculate_accuracy(
        modifyattn, modifyrope, scale_score,
        repeat_paras, num_paraphrases):
    filename = get_filenames(modifyattn, modifyrope, scale_score, repeat_paras, True, num_paraphrases, num_samples=5)
    if os.path.exists(filename) is False:
        print(f"File {filename} does not exist!")
        return None
    df = pandas.read_feather(filename)
    label = f"MyriadLlama {'+Attn' if modifyattn else ''} {'+Rope' if modifyrope else ''} {'+RepeatParas' if repeat_paras else ''} {num_paraphrases} Paras"
    _calculate_accuracy(df, label)
    return df

def report_accuracy(modifyattn, modifyrope, scale_score, repeat_paras):
    calculate_accuracy(modifyattn=modifyattn, modifyrope=modifyrope, scale_score=scale_score, repeat_paras=repeat_paras, num_paraphrases=2)
    calculate_accuracy(modifyattn=modifyattn, modifyrope=modifyrope, scale_score=scale_score, repeat_paras=repeat_paras, num_paraphrases=3)
    calculate_accuracy(modifyattn=modifyattn, modifyrope=modifyrope, scale_score=scale_score, repeat_paras=repeat_paras, num_paraphrases=4)
    calculate_accuracy(modifyattn=modifyattn, modifyrope=modifyrope, scale_score=scale_score, repeat_paras=repeat_paras, num_paraphrases=5)
    # return para2, para3, para4, para5

In [33]:
baseline_fn = f"{dataset_root}/{ds_name}/{model_name}/baseline_per_prompt.feather"
baseline = pandas.read_feather(baseline_fn)
baseline["predict_lemma"] = baseline["predict_lemma"].apply(lambda xs: xs[0])
_calculate_accuracy(baseline, "MyriadLlama Baseline Per Prompt")

MyriadLlama Baseline Per Prompt Accuracy: 0.4578


In [34]:
print("=== No attention or rope modifications, single QA section ===")
modifyattn, modifyrope, repeat_paras = False, False, False
print("--- Without score scaling ---")
report_accuracy(modifyattn, modifyrope, scale_score=False, repeat_paras=repeat_paras)
print("--- With score scaling ---")
report_accuracy(modifyattn, modifyrope, scale_score=True, repeat_paras=repeat_paras)
print()

print("=== With only attention modifications, single QA section ===")
modifyattn, modifyrope, repeat_paras = True, False, False
print("--- Without score scaling ---")
report_accuracy(modifyattn, modifyrope, scale_score=False, repeat_paras=repeat_paras)
print("--- With score scaling ---")
report_accuracy(modifyattn, modifyrope, scale_score=True, repeat_paras=repeat_paras)
print()


print("=== With attention and rope modifications, single QA section ===")
modifyattn, modifyrope, repeat_paras = True, True, False
print("--- Without score scaling ---")
report_accuracy(modifyattn, modifyrope, scale_score=False, repeat_paras=repeat_paras)
print("--- With score scaling ---")
report_accuracy(modifyattn, modifyrope, scale_score=True, repeat_paras=repeat_paras)
print()

=== No attention or rope modifications, single QA section ===
--- Without score scaling ---
MyriadLlama    2 Paras Accuracy: 0.3990
MyriadLlama    3 Paras Accuracy: 0.4285
MyriadLlama    4 Paras Accuracy: 0.4411
MyriadLlama    5 Paras Accuracy: 0.4505
--- With score scaling ---
File /home/xzhao/workspace/GYB_self-ensemble/datasets/myriadlama/qwen2.5_7b_it/myriadlama.scalescore20.singleparaqapair.5samples.2paras.feather does not exist!
File /home/xzhao/workspace/GYB_self-ensemble/datasets/myriadlama/qwen2.5_7b_it/myriadlama.scalescore20.singleparaqapair.5samples.3paras.feather does not exist!
File /home/xzhao/workspace/GYB_self-ensemble/datasets/myriadlama/qwen2.5_7b_it/myriadlama.scalescore20.singleparaqapair.5samples.4paras.feather does not exist!
MyriadLlama    5 Paras Accuracy: 0.1467

=== With only attention modifications, single QA section ===
--- Without score scaling ---
MyriadLlama +Attn   2 Paras Accuracy: 0.3834
MyriadLlama +Attn   3 Paras Accuracy: 0.4111
MyriadLlama +Attn  

In [35]:
print("=== No rope modifications or score scaling, single QA section ===")
scale_score, modifyrope, repeat_paras = False, False, False
print("--- Without attention modifications ---")
report_accuracy(False, modifyrope=modifyrope, scale_score=scale_score, repeat_paras=repeat_paras)
print("--- With attention modifications ---")
report_accuracy(True, modifyrope=modifyrope, scale_score=scale_score, repeat_paras=repeat_paras)

=== No rope modifications or score scaling, single QA section ===
--- Without attention modifications ---
MyriadLlama    2 Paras Accuracy: 0.3990
MyriadLlama    3 Paras Accuracy: 0.4285
MyriadLlama    4 Paras Accuracy: 0.4411
MyriadLlama    5 Paras Accuracy: 0.4505
--- With attention modifications ---
MyriadLlama +Attn   2 Paras Accuracy: 0.3834
MyriadLlama +Attn   3 Paras Accuracy: 0.4111
MyriadLlama +Attn   4 Paras Accuracy: 0.4198
MyriadLlama +Attn   5 Paras Accuracy: 0.4211


In [36]:
print("=== With attention modifications but no score scaling, single QA section ===")
modifyattn, scale_score, repeat_paras = True, False, False
print("--- Without rope modifications ---")
report_accuracy(modifyattn=modifyattn, modifyrope=False, scale_score=scale_score, repeat_paras=repeat_paras)
print("--- With rope modifications ---")
report_accuracy(modifyattn=modifyattn, modifyrope=True, scale_score=scale_score, repeat_paras=repeat_paras)

=== With attention modifications but no score scaling, single QA section ===
--- Without rope modifications ---
MyriadLlama +Attn   2 Paras Accuracy: 0.3834
MyriadLlama +Attn   3 Paras Accuracy: 0.4111
MyriadLlama +Attn   4 Paras Accuracy: 0.4198
MyriadLlama +Attn   5 Paras Accuracy: 0.4211
--- With rope modifications ---
MyriadLlama +Attn +Rope  2 Paras Accuracy: 0.3904
MyriadLlama +Attn +Rope  3 Paras Accuracy: 0.4272
MyriadLlama +Attn +Rope  4 Paras Accuracy: 0.4439
MyriadLlama +Attn +Rope  5 Paras Accuracy: 0.4545


In [37]:
answers = [answers.tolist() for answers in para2["answer_lemmas"].tolist()]
uuids = para2['uuid'].tolist()
pred2, prompt2 = para2['predict_lemma'].tolist(), para2['paraphrases'].tolist()
pred3, prompt3 = para3['predict_lemma'].tolist(), para3['paraphrases'].tolist()
pred4, prompt4 = para4['predict_lemma'].tolist(), para4['paraphrases'].tolist()
pred5, prompt5 = para5['predict_lemma'].tolist(), para5['paraphrases'].tolist()

NameError: name 'para2' is not defined

In [ ]:
# for ans, p0, p2, p3, p4, p5 in zip(answers, base, pred2, pred3, pred4, pred5):
iterator = zip(uuids, answers, pred2, prompt2, pred3, prompt3, pred4, prompt4, pred5, prompt5)
for idx, (uuid, ans, p2, pp2, p3, pp3, p4, pp4, p5, pp5) in enumerate(iterator):
    ans = [a.tolist() for a in ans]
    baseline_sdf = baseline[baseline["uuid"] == uuid]
    prompt = baseline_sdf["paraphrase"].tolist()[0]
    pp2 = pp2.tolist()
    pp3 = pp3.tolist()
    pp4 = pp4.tolist()
    pp5 = pp5.tolist()
    
    pp2_sep_predicts = [baseline_sdf[baseline_sdf["paraphrase"] == pp]["predict_lemma"].tolist()[0] for pp in pp2]
    pp3_sep_predicts = [baseline_sdf[baseline_sdf["paraphrase"] == pp]["predict_lemma"].tolist()[0] for pp in pp3]
    pp4_sep_predicts = [baseline_sdf[baseline_sdf["paraphrase"] == pp]["predict_lemma"].tolist()[0] for pp in pp4]
    pp5_sep_predicts = [baseline_sdf[baseline_sdf["paraphrase"] == pp]["predict_lemma"].tolist()[0] for pp in pp5]
    print("===")
    print("Question:", prompt)
    print("Answers:", ans)
    print("Per-prompt predictions:", baseline_sdf["predict_lemma"].tolist())
    # print("Baseline Prediction:", p0)
    print(f"2-para Prediction:\t'{p2}' from: {pp2_sep_predicts}")
    print(f"3-para Prediction:\t'{p3}' from: {pp3_sep_predicts}")
    print(f"4-para Prediction:\t'{p4}' from: {pp4_sep_predicts}")
    print(f"5-para Prediction:\t'{p5}' from: {pp5_sep_predicts}")
    print()

===
Question: This Is Your Life premiered on the network [MASK].
Answers: [['nbc'], ['national', 'broadcasting', 'company']]
Per-prompt predictions: ['nbc', 'bbc', 'bbc', 'nbc', 'radio', '1955', 'bbc', '1952', 'bbc', '1955']
2-para Prediction:	nbc from: ['nbc', 'bbc']
3-para Prediction:	bbc from: ['nbc', 'bbc', 'radio']
4-para Prediction:	bbc from: ['nbc', 'bbc', '1955', 'radio']
5-para Prediction:	bbc from: ['nbc', 'bbc', 'radio', 'nbc', 'bbc']

===
Question: This Is Your Life premiered on the network [MASK].
Answers: [['nbc'], ['national', 'broadcasting', 'company']]
Per-prompt predictions: ['nbc', 'bbc', 'bbc', 'nbc', 'radio', '1955', 'bbc', '1952', 'bbc', '1955']
2-para Prediction:	1952 from: ['bbc', '1955']
3-para Prediction:	bbc from: ['bbc', '1955', 'bbc']
4-para Prediction:	bbc from: ['nbc', 'radio', '1955', 'bbc']
5-para Prediction:	bbc from: ['bbc', 'nbc', 'radio', 'bbc', '1955']

===
Question: This Is Your Life premiered on the network [MASK].
Answers: [['nbc'], ['national',

'What is the original language of Uspekhi Fizicheskikh Nauk? [MASK].'